In [0]:
# ==============================================================================
# Pipeline Step: 02_silver_to_gold.py
# Description: Reads cleaned Delta tables from the Silver layer, calculates key business
#              metrics and aggregations, and writes the output as Gold Delta tables.
# ==============================================================================

from pyspark.sql.functions import avg, col, count, current_timestamp, date_format, round, sum

# ------------------------------------------------------------------------------
# 1. Storage Credentials & Container Paths
# ------------------------------------------------------------------------------
storage_account = "sttransitanalyticsdev"
storage_key = "kcLG0+ay1Ff8B2BXabvChxvASTlgkEiCwXMGdb4cjbEq3WanFq/u3uvrh9IldGJzwQbsgLaMwo+7+ASt/+3sJQ=="

# ABFSS protocol URIs for Silver and Gold containers
SILVER_PATH = f"abfss://silver@{storage_account}.dfs.core.windows.net"
GOLD_PATH = f"abfss://gold@{storage_account}.dfs.core.windows.net"

# Package Azure storage authentication configuration
storage_options = {
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net": storage_key
}

# ------------------------------------------------------------------------------
# 2. Ingest Silver Delta Tables
# ------------------------------------------------------------------------------
df_trips = (
    spark.read.options(**storage_options)
    .format("delta")
    .load(f"{SILVER_PATH}/trips")
)

df_routes = (
    spark.read.options(**storage_options)
    .format("delta")
    .load(f"{SILVER_PATH}/routes")
)

df_buses = (
    spark.read.options(**storage_options)
    .format("delta")
    .load(f"{SILVER_PATH}/buses")
)

df_payments = (
    spark.read.options(**storage_options)
    .format("delta")
    .load(f"{SILVER_PATH}/payments")
)

print("✅ Silver Delta tables successfully loaded into memory.")

# ------------------------------------------------------------------------------
# 3. Gold Aggregations & Table Generation
# ------------------------------------------------------------------------------

# --- 3.1 Gold Table: Route Performance Analysis ---
# Computes total trips, passenger counts, total revenue, and average ridership per route.
gold_route_performance = (
    df_trips.join(df_routes, "route_id", "inner")
    .join(df_payments, "trip_id", "left")
    .groupBy("route_id", "route_name", "origin", "destination")
    .agg(
        count("trip_id").alias("total_trips"),
        sum("passenger_count").alias("total_passengers"),
        round(sum("amount"), 2).alias("total_revenue"),
        round(avg("passenger_count"), 1).alias("avg_passengers_per_trip"),
    )
    .withColumn("created_at", current_timestamp())
)

(
    gold_route_performance.write.options(**storage_options)
    .format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/gold_route_performance")
)

print("🏆 Gold Table created: gold_route_performance")


# --- 3.2 Gold Table: Bus Utilization & Capacity Efficiency ---
# Measures total trips executed and capacity usage percentages per vehicle.
gold_bus_utilization = (
    df_trips.join(df_buses, "bus_id", "inner")
    .groupBy("bus_id", "plate_num", "capacity")
    .agg(
        count("trip_id").alias("trips_completed"),
        sum("passenger_count").alias("total_passengers_carried"),
        round(avg("passenger_count") / col("capacity") * 100, 2).alias(
            "avg_capacity_utilization_pct"
        ),
    )
    .withColumn("created_at", current_timestamp())
)

(
    gold_bus_utilization.write.options(**storage_options)
    .format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/gold_bus_utilization")
)

print("🏆 Gold Table created: gold_bus_utilization")


# --- 3.3 Gold Table: Daily Revenue Trends ---
# Summarizes financial transactions grouped by date and payment method.
gold_daily_revenue = (
    df_payments.withColumn(
        "payment_date", date_format(col("payment_timestamp"), "yyyy-MM-dd")
    )
    .groupBy("payment_date", "payment_method")
    .agg(
        sum("amount").alias("daily_revenue"),
        count("payment_id").alias("transaction_count"),
    )
    .withColumn("created_at", current_timestamp())
)

(
    gold_daily_revenue.write.options(**storage_options)
    .format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/gold_daily_revenue")
)

print("🏆 Gold Table created: gold_daily_revenue")

✅ Silver Delta tables successfully loaded into memory.
🏆 Gold Table created: gold_route_performance
🏆 Gold Table created: gold_bus_utilization
🏆 Gold Table created: gold_daily_revenue
